# 10 — Minimax evasion and Alpha-Beta

Minimax looks ahead as the drone and interceptor take turns. The drone tries to move toward a safe temporary waypoint while increasing its distance from capture; the interceptor tries to close the gap. Alpha-Beta prunes branches but returns the same move and value.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
from uavplanner.adversarial import choose_move
from uavplanner.config import Config
from uavplanner.console import Console
from uavplanner.models import Drone, EnergyCoef, Interceptor, Target, World
from uavplanner.planner import Plan
from uavplanner.simulator import run_mission

mask = np.zeros((9, 9), dtype=bool)
plain = choose_move((4, 4), (4, 2), (0, 8), mask, use_alpha_beta=False)
pruned = choose_move((4, 4), (4, 2), (0, 8), mask, use_alpha_beta=True)
assert plain[:2] == pruned[:2]
print(f'move={pruned[0]}, value={pruned[1]:.1f}, nodes={plain[2]} -> {pruned[2]}')

## R6 through the simulator

This small scenario sets the interceptor's detection radius to five cells. The drone starts at the R6 trigger distance, evades the approaching patrol, then resumes its target route after the threat is clear. The smaller detection radius makes the moment the threat is first observed easy to see; it is a scenario setting, not the project default.

In [ ]:
grid_n = 21
base = (10, 10)
world = World(np.zeros((grid_n, grid_n)), np.zeros((grid_n, grid_n), dtype=bool), (0.0, 0.0), base)
target = Target(id=1, pos=(10, 15), mass_kg=0.5)
drone = Drone(id=1, battery=150.0, pos=base)
route = [(10, col) for col in range(10, 16)] + [(10, col) for col in range(14, 9, -1)]
plan = Plan('GO', [], {}, {1: 1}, {1: [1]}, {1: route}, {1: 10.0})
interceptor = Interceptor(pos=(10, 5), waypoints=[(0, 0)])
cfg = Config(grid_n=grid_n, max_ticks=60, takeoff_stagger=0, noise_sd=0.0, interceptor_detection=5)
console = Console()
summary = run_mission(world, plan, [drone], interceptor, [target], EnergyCoef(), EnergyCoef(), cfg, np.random.default_rng(2), console, [])
assert summary['collisions'] == 0
assert summary['minimax_nodes'] > 0
assert summary['drones_lost'] == 0
assert summary['targets_confirmed'] == 1
print(summary)
for row in console.rows():
    if row.event:
        print(row.tick, row.drone, row.pos, row.mode, row.event)

## What to notice

`R6: EVADE` marks the Minimax handoff. When separation exceeds `cfg.minimax_release`, the simulator rebuilds an A* route through remaining targets and back to base. Capture indices returned by the interceptor mark drones LOST; this seeded run completes without capture.